<a href="https://colab.research.google.com/github/manishsaini15/Encoder_Decoder_Concepts/blob/main/Encoder_Decoder_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

# Input sentence(English) -> Encoder -> Internal memory/meaning -> Decoder -> Output sentence(Hindi)
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [4]:
# ============================================================
# 2. CREATE A SMALL TOY DATASET
# ============================================================
#
# input_texts  -> English sentences
# target_texts -> Hindi sentences
#
# We add special tokens in the target sentence:
#   <start>  -> tells decoder where to begin
#   <end>    -> tells decoder where to stop
#
# Example target:
#   <start> मुझे चाय पसंद है <end>
#
# During training, decoder learns:
#   input  = <start> मुझे चाय पसंद है
#   target = मुझे चाय पसंद है <end>
#
# This is called TEACHER FORCING.
# ============================================================


input_texts = [
    "i am happy",
    "i am sad",
    "he is good",
    "she is good",
    "i love tea",
    "i love coffee",
    "he likes mango",
    "she likes music",
    "i eat rice",
    "he drinks water"
]

target_texts = [
    "<start> मैं खुश हूँ <end>",
    "<start> मैं दुखी हूँ <end>",
    "<start> वह अच्छा है <end>",
    "<start> वह अच्छी है <end>",
    "<start> मुझे चाय पसंद है <end>",
    "<start> मुझे कॉफी पसंद है <end>",
    "<start> उसे आम पसंद है <end>",
    "<start> उसे संगीत पसंद है <end>",
    "<start> मैं चावल खाता हूँ <end>",
    "<start> वह पानी पीता है <end>"
]

In [5]:
# ============================================================
# 3. TOKENIZE INPUT AND TARGET SENTENCES
# ============================================================
#
# Neural networks do not work directly with text.
# So we convert words into integers.
#
# Example:
#   "i love tea" -> [1, 5, 8]
#
# We use TWO separate tokenizers:
#   - input_tokenizer  for English
#   - target_tokenizer for Hindi
#
# Very important:
# We use filters='' so that special tokens like
# <start> and <end> remain exactly as they are.
# ============================================================

input_tokenizer = Tokenizer(filters='') #by default tokenizer remove punctuation-like(<) character <start> ---> start
input_tokenizer.fit_on_texts(input_texts)

# {'i': 1, 'am': 2, 'happy': 3}

target_tokenizer = Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_texts)

#"'<start>': 1, 'मैं': 2, 'खुश': 3, 'हूँ':4

# Convert sentences into sequences of integers
input_sequences = input_tokenizer.texts_to_sequences(input_texts)   # I am happy -> [1,2,3]
target_sequences = target_tokenizer.texts_to_sequences(target_texts) #<start> म ै खुश हूँ <end>-> [1,2,3,4,5]

# Vocabulary size = number of unique words + 1
# +1 because token id 0 is reserved for padding
input_vocab_size = len(input_tokenizer.word_index) + 1
target_vocab_size = len(target_tokenizer.word_index) + 1

print(input_vocab_size)

# Reverse mapping: integer -> word
# This helps during prediction, so we can convert
# predicted token ids back to words
reverse_input_word_index = {index: word for word, index in input_tokenizer.word_index.items()}
reverse_target_word_index = {index: word for word, index in target_tokenizer.word_index.items()}

print("============================================================")
print("INPUT WORD INDEX")
print("============================================================")
print(input_tokenizer.word_index)
print()

print("============================================================")
print("TARGET WORD INDEX")
print("============================================================")
print(target_tokenizer.word_index)
print()


19
INPUT WORD INDEX
{'i': 1, 'he': 2, 'am': 3, 'is': 4, 'good': 5, 'she': 6, 'love': 7, 'likes': 8, 'happy': 9, 'sad': 10, 'tea': 11, 'coffee': 12, 'mango': 13, 'music': 14, 'eat': 15, 'rice': 16, 'drinks': 17, 'water': 18}

TARGET WORD INDEX
{'<start>': 1, '<end>': 2, 'है': 3, 'पसंद': 4, 'मैं': 5, 'हूँ': 6, 'वह': 7, 'मुझे': 8, 'उसे': 9, 'खुश': 10, 'दुखी': 11, 'अच्छा': 12, 'अच्छी': 13, 'चाय': 14, 'कॉफी': 15, 'आम': 16, 'संगीत': 17, 'चावल': 18, 'खाता': 19, 'पानी': 20, 'पीता': 21}



In [6]:
# ============================================================
# 4. FIND MAXIMUM SEQUENCE LENGTHS
# ============================================================
#
# In one batch, all sequences must have same length.
# So we need to know the maximum sentence length.
#
# Example:
#   [1, 2, 3]
#   [4, 5]
#
# After padding:
#   [1, 2, 3]
#   [4, 5, 0]
# ============================================================

max_encoder_seq_length = max(len(seq) for seq in input_sequences)
max_decoder_seq_length = max(len(seq) for seq in target_sequences)

print("Max encoder sequence length:", max_encoder_seq_length)
print("Max decoder sequence length:", max_decoder_seq_length)
print()


Max encoder sequence length: 3
Max decoder sequence length: 6



In [7]:
# ============================================================
# 5. PAD ENCODER INPUT SEQUENCES
# ============================================================
#
# Post-padding means zeros are added at the end.
#
# Example:
#   [1, 2] -> [1, 2, 0, 0]
# ============================================================

encoder_input_data = pad_sequences(
    input_sequences,
    maxlen=max_encoder_seq_length,
    padding='post'
)

In [9]:
# ============================================================
# 6. PREPARE DECODER INPUT DATA AND DECODER TARGET DATA
# ============================================================
#
# This is a very important concept.
#
# Suppose target sentence is:
#   <start> मैं खुश हूँ <end>
#
# Then:
#   decoder input  = <start> मैं खुश हूँ
#   decoder target = मैं खुश हूँ <end>
#
# Why?
# Because decoder learns to predict NEXT word.
#
# Training pairs become:
#   <start> -> मैं
#   मैं      -> खुश
#   खुश      -> हूँ
#   हूँ       -> <end>
#
# ============================================================
decoder_input_sequences = []
decoder_target_sequences = []

for seq in target_sequences:
    # All tokens except the last one go to decoder input
    decoder_input_sequences.append(seq[:-1])

    # All tokens except the first one go to decoder target
    decoder_target_sequences.append(seq[1:])

# Pad decoder input
decoder_input_data = pad_sequences(
    decoder_input_sequences,
    maxlen=max_decoder_seq_length - 1,
    padding='post'
)

# Pad decoder target
decoder_target_data = pad_sequences(
    decoder_target_sequences,
    maxlen=max_decoder_seq_length - 1,
    padding='post'
)


In [10]:
# For sparse_categorical_crossentropy, the target should have shape:
#   (number_of_samples, sequence_length, 1)
#
# So we expand the last dimension
decoder_target_data = np.expand_dims(decoder_target_data, -1)

print("Encoder input data shape :", encoder_input_data.shape)
print("Decoder input data shape :", decoder_input_data.shape)
print("Decoder target data shape:", decoder_target_data.shape)
print()


Encoder input data shape : (10, 3)
Decoder input data shape : (10, 5)
Decoder target data shape: (10, 5, 1)



In [11]:
# ============================================================
# 7. DEFINE MODEL HYPERPARAMETERS
# ============================================================
#
# embedding_dim = dimension of each word embedding vector
# latent_dim    = number of units in LSTM hidden state
#
# These values are small because this is only a toy example.
# ============================================================

embedding_dim = 64
latent_dim = 128



In [14]:
# ============================================================
# 8. BUILD THE ENCODER
# ============================================================
#
# Encoder workflow:
#   Input sentence
#      -> Embedding
#      -> LSTM
#      -> final hidden state + final cell state
#
# These final states represent the encoded meaning
# of the input sentence.
# ============================================================

# Encoder input layer
encoder_inputs = Input(shape=(None,), name="encoder_inputs")

# Embedding layer converts integer tokens to dense vectors
encoder_embedding = Embedding(
    input_dim=input_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,              # ignore padding token 0
    name="encoder_embedding"
)(encoder_inputs)

# tea -> [0.12,0.31,044.....]
# Encoder LSTM
# return_state=True gives:
#   state_h = final hidden state
#   state_c = final cell state
encoder_lstm = LSTM(
    latent_dim,
    return_state=True,
    name="encoder_lstm"
)

# encoder_outputs is not used in this basic model
# because attention is not used yet
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

# Save encoder states to pass to decoder
encoder_states = [state_h, state_c]

In [15]:
# ============================================================
# 9. BUILD THE DECODER
# ============================================================
#
# Decoder workflow:
#   Decoder input sentence
#      -> Embedding
#      -> LSTM initialized with encoder states
#      -> Dense layer
#      -> probability distribution over target vocabulary
#
# Decoder predicts one target token at each time step.
# ============================================================

# Decoder input layer
decoder_inputs = Input(shape=(None,), name="decoder_inputs")

# Decoder embedding layer
decoder_embedding_layer = Embedding(
    input_dim=target_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="decoder_embedding"
)

decoder_embedding = decoder_embedding_layer(decoder_inputs)

In [16]:
# Decoder LSTM
# return_sequences=True because decoder outputs one vector
# for each time step
# return_state=True because we will later reuse states during inference
decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

# input <start> -> output मुझे
#......

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

# Dense layer converts decoder LSTM output to probabilities
# over all target vocabulary words
decoder_dense = Dense(
    target_vocab_size,
    activation='softmax',
    name="decoder_output_dense"
)

# मुझे : 0.7
# चाय: 0.05


decoder_outputs = decoder_dense(decoder_outputs)

In [17]:
# ============================================================
# 10. BUILD THE FULL TRAINING MODEL
# ============================================================
#
# Inputs:
#   - encoder input sequence
#   - decoder input sequence
#
# Output:
#   - decoder predicted sequence
#
# Loss:
#   sparse_categorical_crossentropy
# because target is integer token id, not one-hot vector
# ============================================================

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("============================================================")
print("MODEL SUMMARY")
print("============================================================")
model.summary()
print()

MODEL SUMMARY


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 64)  │      1,216 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 64)  │      1,408 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │     98,816 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal_1[0][0] │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     98,816 │ decoder_embeddin… │
│                     │ 128), (None,      │            │ encoder_lstm[0][… │
│                     │ 128), (None,      │            │ encoder_lstm[0][… │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output_den… │ (None, None, 22)  │      2,838 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 203,094 (793.34 KB)

 Trainable params: 203,094 (793.34 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# ============================================================
# 11. TRAIN THE MODEL
# ============================================================
#
# Since dataset is tiny, we train for many epochs.
# This helps the model memorize the small dataset.
#
# In real-world tasks:
#   - data would be much larger
#   - training would be more advanced
# ============================================================

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=300,
    verbose=1
)

# 1) model predicts target token
# 2) compare prediction with the actual target
# 3) compute error
# 4) optimize/change weights
# 5) repeat over many epochs


Epoch 1/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.1739 - loss: 3.0827
Epoch 2/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2174 - loss: 3.0481
Epoch 3/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2174 - loss: 2.9960
Epoch 4/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2174 - loss: 2.8997
Epoch 5/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2174 - loss: 2.6694
Epoch 6/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2174 - loss: 2.3781
Epoch 7/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2174 - loss: 2.3374
Epoch 8/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.2609 - loss: 2.2435
Epoch 9/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3261 - loss: 2.1693
Epoch 10/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2174 - loss: 2.1092
Epoch 11/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2174 - loss: 2.0570
Epoch 12/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3261 - lo

In [20]:
# ============================================================
# 12. BUILD ENCODER INFERENCE MODEL
# ============================================================
#
# Training model and prediction model are different.
#
# During training:
#   entire decoder input sequence is given at once
#
# During prediction:
#   decoder predicts one token at a time
#
# So we need a separate encoder model for inference.
#
# Encoder inference model:
#   input sentence -> encoder states
# ============================================================

encoder_model = Model(
    encoder_inputs,
    encoder_states
)


In [21]:
# ============================================================
# 13. BUILD DECODER INFERENCE MODEL
# ============================================================
#
# During inference, decoder needs:
#   - current input token
#   - previous hidden state
#   - previous cell state
#
# It returns:
#   - next token probabilities
#   - updated hidden state
#   - updated cell state
#
# We reuse the same trained decoder layers.
# ============================================================

# Inputs for decoder states during inference
decoder_state_input_h = Input(shape=(latent_dim,), name="decoder_state_input_h")
decoder_state_input_c = Input(shape=(latent_dim,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Reuse decoder embedding layer
decoder_inf_embedding = decoder_embedding_layer(decoder_inputs)

# Reuse decoder LSTM layer
decoder_inf_outputs, state_h_inf, state_c_inf = decoder_lstm(
    decoder_inf_embedding,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h_inf, state_c_inf]

# Reuse decoder dense layer
decoder_inf_outputs = decoder_dense(decoder_inf_outputs)

# Final decoder model for inference
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_inf_outputs] + decoder_states
)


In [22]:
# ============================================================
# 14. DECODE FUNCTION
# ============================================================
#
# This function performs actual prediction.
#
# Steps:
#   1. Encode the input sentence
#   2. Start decoder with <start> token
#   3. Predict one token
#   4. Feed predicted token back to decoder
#   5. Continue until <end> token is predicted
#
# ============================================================

def decode_sequence(input_seq):
    """
    Convert one encoded input sentence into target sentence.

    Parameters:
        input_seq : padded input sequence of shape (1, max_encoder_seq_length)

    Returns:
        decoded sentence as a string
    """

    # Step 1: Encode the input sequence to get initial decoder states
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Step 2: Get token ids for <start> and <end>
    # Since we used Tokenizer(filters=''),
    # these special tokens remain exactly as "<start>" and "<end>"
    start_token_index = target_tokenizer.word_index["<start>"]
    end_token_index = target_tokenizer.word_index["<end>"]

    # Step 3: First decoder input is always <start>
    target_seq = np.array([[start_token_index]])

    # We will collect predicted words here
    decoded_words = []

    # Step 4: Predict one word at a time
    for _ in range(max_decoder_seq_length):

        # Decoder predicts probability distribution for next token
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        # output_tokens shape:
        #   (1, 1, target_vocab_size)
        #
        # We choose the token with highest probability
        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        # If model predicts padding token, stop
        if sampled_token_index == 0:
            break

        # Convert token id to actual word
        sampled_word = reverse_target_word_index.get(sampled_token_index, "")

        # If <end> token comes, stop generation
        if sampled_token_index == end_token_index:
            break

        # Save the predicted word
        decoded_words.append(sampled_word)

        # The predicted token becomes next input to decoder
        target_seq = np.array([[sampled_token_index]])

        # Update decoder states
        states_value = [h, c]

    # Join all predicted words into one sentence
    return " ".join(decoded_words)


In [23]:
# ============================================================
# 15. TRANSLATE RAW SENTENCE FUNCTION
# ============================================================
#
# This helper function:
#   - takes raw English text
#   - converts it to token sequence
#   - pads it
#   - sends it for decoding
#
# ============================================================

def translate_sentence(sentence):
    """
    Translate a raw input English sentence into Hindi.

    Parameters:
        sentence : string

    Returns:
        translated Hindi sentence as string
    """

    # Convert sentence into token sequence
    seq = input_tokenizer.texts_to_sequences([sentence])

    # Pad the sequence
    seq = pad_sequences(
        seq,
        maxlen=max_encoder_seq_length,
        padding='post'
    )

    # Decode sequence
    prediction = decode_sequence(seq)
    return prediction


In [24]:
# ============================================================
# 16. TEST ON TRAINING SENTENCES
# ============================================================
#
# Since training data is tiny, the model should perform
# well on these seen examples after enough epochs.
# ============================================================

print("\n============================================================")
print("PREDICTIONS ON TRAINING DATA")
print("============================================================\n")

for i, sentence in enumerate(input_texts):
    predicted = translate_sentence(sentence)
    print(f"Input      : {sentence}")
    print(f"Predicted  : {predicted}")
    print(f"Expected   : {target_texts[i]}")
    print("-" * 60)


PREDICTIONS ON TRAINING DATA

Input      : i am happy
Predicted  : मैं खुश हूँ
Expected   : <start> मैं खुश हूँ <end>
------------------------------------------------------------
Input      : i am sad
Predicted  : मैं दुखी हूँ
Expected   : <start> मैं दुखी हूँ <end>
------------------------------------------------------------
Input      : he is good
Predicted  : वह अच्छा है
Expected   : <start> वह अच्छा है <end>
------------------------------------------------------------
Input      : she is good
Predicted  : वह अच्छी है
Expected   : <start> वह अच्छी है <end>
------------------------------------------------------------
Input      : i love tea
Predicted  : मुझे चाय पसंद है
Expected   : <start> मुझे चाय पसंद है <end>
------------------------------------------------------------
Input      : i love coffee
Predicted  : मुझे कॉफी पसंद है
Expected   : <start> मुझे कॉफी पसंद है <end>
------------------------------------------------------------
Input      : he likes mango
Predicted  : उसे आम प

In [25]:
# ============================================================
# 17. TEST ON SOME CUSTOM SENTENCES
# ============================================================
#
# These are just sample tests.
# Since the dataset is tiny, the model may only work
# properly on sentences similar to training examples.
# ============================================================

print("\n============================================================")
print("CUSTOM TESTS")
print("============================================================\n")

test_sentences = [
    "i love tea",
    "i love coffee",
    "he likes mango",
    "she likes music",
    "i am happy",
    "he drinks water"
]

for sentence in test_sentences:
    output = translate_sentence(sentence)
    print(f"Input  : {sentence}")
    print(f"Output : {output}")
    print("-" * 60)


CUSTOM TESTS

Input  : i love tea
Output : मुझे चाय पसंद है
------------------------------------------------------------
Input  : i love coffee
Output : मुझे कॉफी पसंद है
------------------------------------------------------------
Input  : he likes mango
Output : उसे आम पसंद है
------------------------------------------------------------
Input  : she likes music
Output : उसे संगीत पसंद है
------------------------------------------------------------
Input  : i am happy
Output : मैं खुश हूँ
------------------------------------------------------------
Input  : he drinks water
Output : वह पानी पीता है
------------------------------------------------------------


In [26]:
# ============================================================
# 18. OPTIONAL: INTERACTIVE TESTING
# ============================================================
#
# You can uncomment this block if you want to allow
# user input from keyboard.
# ============================================================


print("\nType 'exit' to stop.")
while True:
    user_input = input("Enter English sentence: ").strip().lower()
    if user_input == "exit":
        break
    print("Translation:", translate_sentence(user_input))
    print()



Type 'exit' to stop.
Enter English sentence: I am going to jaipur
Translation: मैं खुश

Enter English sentence: exit
